# GCMC Data Generation for cDFT Neural Operators

This tutorial generates training data for the three learnable operators:

| Operator | Mapping | Trained from |
|---|---|---|
| **G₁** | V_ext(r) → ρ(r) | (v_ext, rho) pairs |
| **G₂** | ρ(r) → c⁽¹⁾(r) = δF_exc/δρ | (rho, c1) pairs — the Sammüller target |
| **G₃** | c⁽¹⁾(r) → F_exc | thermodynamic integration |

**References**
- Sammüller et al., *J. Phys.: Condens. Matter* **36**, 243002 (2024)
- Evans et al., *Phys. Rev. Lett.* **134**, 148001 (2025)

**Two sections:**
1. CPU baseline — reproduce the Sammüller random-V_ext protocol with an analytic pair potential
2. GPU+MACE-MP — same pipeline with real MLIP energies (runs on `yemba_local` or any CUDA GPU)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Verify imports
from mlip_mc.dft import (
    random_fourier_field, HardWall, CompositeField,
    DensityGrid1D, SampleWriter
)
from mlip_mc.dft.sample_writer import load_sample
from mlip_mc.dft.campaign import GCMCCampaign

print('mlip_mc.dft imports OK')

## Part 1 — CPU Baseline: Sammüller Protocol

Reproduce the data-generation scheme from Sammüller 2024:
- Random Fourier-series V_ext per run
- GCMC with analytic LJ pair potential (no MLIP)
- Output: ρ(z) + c⁽¹⁾(z) per run, saved as HDF5

We use a simple Lennard-Jones calculator as a drop-in so the pipeline can run on any CPU.

In [ ]:
from ase.calculators.lj import LennardJones
from ase import Atoms
import numpy as np

# Minimal framework: empty periodic box (models a bulk fluid between walls)
L = 20.0   # Å  — box size
frame = Atoms(cell=[L, L, L], pbc=True)

# Adsorbate: single-atom argon-like particle
ads = Atoms('Ar', positions=[[0, 0, 0]])

# LJ calculator (ε=0.01 eV, σ=3.4 Å — argon)
lj_calc = LennardJones(epsilon=0.01, sigma=3.4, rc=10.0)

print(f'Box: {L}×{L}×{L} Å, adsorbate: Ar')

In [ ]:
# Visualise a few random V_ext draws — this is the diversity that trains G₁ and G₂

rng = np.random.default_rng(42)
z = np.linspace(0, L, 200)
dummy_pos = np.column_stack([np.zeros(200), np.zeros(200), z])

fig, ax = plt.subplots(figsize=(8, 3))
for _ in range(6):
    v = random_fourier_field(axis=2, n_modes=4, amplitude_scale=0.1, L=L, rng=rng)
    ax.plot(z, v(dummy_pos), alpha=0.7)

ax.set_xlabel('z [Å]')
ax.set_ylabel('V_ext [eV]')
ax.set_title('Random Fourier V_ext draws (Sammüller protocol)')
plt.tight_layout()
plt.show()

In [ ]:
# Run a small campaign: 3 μ values × 5 random V_ext = 15 samples
# (Use n_equil=500, n_prod=2000 for quick testing; scale up for production)

OUTPUT_DIR = Path('./cdft_tutorial_data')

cfg = {
    'temperature': 300.0,
    'mu_values': [-0.3, -0.1, 0.1],
    'n_runs': 5,
    'n_equilibration_steps': 500,
    'n_production_steps': 2000,
    'grid': {'n_bins': 100, 'axis': 2, 'n_blocks': 5},
    'v_ext': {'type': 'fourier', 'n_modes': 4, 'amplitude_scale': 0.1},
    'adsorbent': None,         # will be overridden programmatically
    'adsorbate_molecule': 'Ar',
    'output_dir': str(OUTPUT_DIR),
    'seed': 42,
}

# Use LJ calculator
campaign = GCMCCampaign(config=cfg, model=lj_calc)
# Override the frame directly (bypasses adsorbent file requirement)
campaign.atoms_frame = frame
campaign.box = np.array([L, L, L])

from ase.build import molecule as ase_molecule
campaign.atoms_ads = Atoms('Ar', positions=[[0, 0, 0]])
campaign.adsorbate_name = 'Ar'

written = campaign.run()
print(f'\nWrote {len(written)} samples to {OUTPUT_DIR}')

In [ ]:
# Load and inspect one sample
sample = load_sample(written[0])

fig, axes = plt.subplots(1, 3, figsize=(12, 3))

axes[0].plot(sample['z'], sample['v_ext'])
axes[0].set_title('V_ext(z) [eV]')
axes[0].set_xlabel('z [Å]')

axes[1].plot(sample['z'], sample['rho'])
axes[1].fill_between(sample['z'],
                     sample['rho'] - sample['rho_stderr'],
                     sample['rho'] + sample['rho_stderr'], alpha=0.3)
axes[1].set_title('ρ(z) [Å⁻³]')
axes[1].set_xlabel('z [Å]')

axes[2].plot(sample['z'], sample['c1'])
axes[2].fill_between(sample['z'],
                     sample['c1'] - sample['c1_stderr'],
                     sample['c1'] + sample['c1_stderr'], alpha=0.3)
axes[2].set_title('c⁽¹⁾(z)  [= ln ρ − βμ + βV_ext]')
axes[2].set_xlabel('z [Å]')

plt.suptitle(f"μ = {sample['metadata']['mu']:.2f} eV   T = {sample['metadata']['T']} K")
plt.tight_layout()
plt.show()

print('HDF5 schema:', list(sample.keys()))
print('Metadata:', sample['metadata'])

## Part 2 — GPU+MACE-MP: Real System

Same pipeline, MLIP energies via MACE-MP (Foundation model).  
Runs on `yemba_local` or any CUDA GPU — skipped gracefully on CPU-only machines.

Supports both the Sammüller protocol (varied V_ext, compute ρ+c⁽¹⁾) and
the PRL 134 protocol (two-species ionic fluid, hard-wall confinement).

In [ ]:
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

if DEVICE == 'cpu':
    print('⚠ No GPU found — MACE-MP cells will run on CPU (slow). '
          'Connect to yemba_local for GPU speedup.')

GPU_AVAILABLE = (DEVICE == 'cuda')
if GPU_AVAILABLE:
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Option A: TorchSim batched runner (fastest — N chains in one forward pass)
# Requires: pip install mlip-mc[dft] mace-torch torch-sim-atomistic

try:
    from mlip_mc.dft.gpu import BatchedGCMCRunner, build_torchsim_model

    ts_model = build_torchsim_model('mace-mp', device=DEVICE)

    n_chains = 4
    rng = np.random.default_rng(123)
    v_ext_list = [
        CompositeField([
            HardWall(axis=2, z_lo=0.5, z_hi=L - 0.5, strength=500.0),
            random_fourier_field(axis=2, n_modes=4, amplitude_scale=0.05, L=L, rng=rng),
        ])
        for _ in range(n_chains)
    ]

    runner = BatchedGCMCRunner(
        model=ts_model,
        atoms_frame=frame,
        atoms_ads_list=[Atoms('Ar', positions=[[0, 0, 0]])] * n_chains,
        T_list=[300.0] * n_chains,
        mu_list=[-0.4, -0.2, 0.0, 0.2],
        v_ext_list=v_ext_list,
        n_equil=200,
        n_prod=500,
        n_bins=100,
        n_blocks=5,
        device=DEVICE,
    )

    chain_results = runner.run()
    print(f'\nBatched runner: {len(chain_results)} chain samples collected')

    # Write to HDF5
    from mlip_mc.dft.campaign import _v_ext_to_dict
    import json
    for idx, (r, mu, v_ext) in enumerate(zip(chain_results, [-0.4, -0.2, 0.0, 0.2], v_ext_list)):
        writer = SampleWriter(OUTPUT_DIR / 'mace', run_id=f'mace_run_{idx:03d}')
        writer.write(
            grid_results=r,
            metadata={'T': 300.0, 'mu': mu, 'beta': r['total_steps'],
                      'adsorbate': 'Ar', 'backend': 'mace-mp',
                      'v_ext_spec': json.dumps(_v_ext_to_dict(v_ext)),
                      'box': [L, L, L]},
        )
    print(f'Written to {OUTPUT_DIR}/mace/')

except ImportError as e:
    print(f'TorchSim/MACE not installed ({e}) — skipping GPU section.')
    print('Install with: pip install mlip-mc[dft] mace-torch torch-sim-atomistic')

In [ ]:
# Option B: Standard ASE+MACE-MP via existing run_gcmc() with dft extension
# Slower (one forward per move) but works out of the box with mace-torch installed.

try:
    from mace.calculators import mace_mp
    from mlip_mc.src.gcmc import MLP_GCMC
    from mlip_mc.dft import random_fourier_field, DensityGrid1D
    from ase.units import bar, kB
    from ase.data import vdw_radii

    mace_calc = mace_mp(model='small', device=DEVICE, default_dtype='float64')
    v_ext = CompositeField([
        HardWall(axis=2, z_lo=0.5, z_hi=L - 0.5, strength=500.0),
        random_fourier_field(axis=2, n_modes=4, amplitude_scale=0.05, L=L),
    ])

    T, mu = 300.0, -0.2
    beta = 1.0 / (kB * T)
    P = np.exp(beta * mu) * bar

    grid = DensityGrid1D(box_z=L, n_bins=100, n_blocks=5, xy_area=L**2)

    gcmc = MLP_GCMC(
        model=mace_calc,
        atoms_frame=frame.copy(),
        atoms_ads=Atoms('Ar', positions=[[0, 0, 0]]),
        T=T, P=P, fugacity=P,
        device=DEVICE,
        vdw_radii=vdw_radii,
        output_dir=str(OUTPUT_DIR / 'mace_serial'),
        n_equilibration_steps=200,
        n_production_steps=500,
        external_field=v_ext,
        density_grid=grid,
    )
    gcmc.run(N=700)

    res = grid.results(mu=mu, beta=beta, external_field=v_ext)
    print(f'ρ mean: {res["rho"].mean():.4f} Å⁻³  |  c⁽¹⁾ mean: {res["c1"].mean():.4f}')

except ImportError as e:
    print(f'MACE not installed ({e}) — install with: pip install mace-torch')

## Part 3 — Minimal Neural Functional (MLP-in-window)

Train G₂: ρ(r) → c⁽¹⁾(r) using a local MLP in a sliding window,
following Sammüller 2024 and the NeuralDFT tutorial.

In [ ]:
import torch
import torch.nn as nn
from mlip_mc.dft.sample_writer import load_sample

WINDOW_BINS = 41   # ±20 bins around each position (Sammüller uses ±1σ)

class NeuralFunctionalG2(nn.Module):
    """G₂: local density window → c⁽¹⁾ at centre.  Sammüller 2024 architecture."""
    def __init__(self, window: int = WINDOW_BINS):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(window, 128), nn.Tanh(),
            nn.Linear(128, 64),    nn.Tanh(),
            nn.Linear(64, 32),     nn.Tanh(),
            nn.Linear(32, 1),
        )

    def forward(self, rho_window: torch.Tensor) -> torch.Tensor:
        return self.net(rho_window).squeeze(-1)


def make_windows(rho: np.ndarray, c1: np.ndarray, w: int = WINDOW_BINS):
    """Extract sliding windows and corresponding c1 values."""
    half = w // 2
    n = len(rho)
    rho_padded = np.pad(rho, half, mode='wrap')
    X = np.stack([rho_padded[i: i + w] for i in range(n)], axis=0)
    y = c1
    # Data augmentation: mirror symmetry (Sammüller trick)
    X = np.concatenate([X, X[:, ::-1].copy()], axis=0)
    y = np.concatenate([y, y], axis=0)
    return X.astype(np.float32), y.astype(np.float32)


# Load samples from Part 1
sample_files = sorted(Path('./cdft_tutorial_data').glob('run_*.h5'))
print(f'Found {len(sample_files)} samples')

X_all, y_all = [], []
for f in sample_files:
    s = load_sample(f)
    # Skip bins where density is below threshold (c1 is unreliable)
    mask = s['rho'] > 1e-6
    if mask.sum() < WINDOW_BINS:
        continue
    X, y = make_windows(s['rho'], s['c1'])
    X_all.append(X)
    y_all.append(y)

if X_all:
    X_all = np.concatenate(X_all, axis=0)
    y_all = np.concatenate(y_all, axis=0)
    print(f'Training set: {X_all.shape[0]} windows × {X_all.shape[1]} features')
else:
    print('No samples yet — run Part 1 first')

In [ ]:
if 'X_all' in dir() and len(X_all) > 0:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model_g2 = NeuralFunctionalG2(WINDOW_BINS).to(device)
    opt = torch.optim.Adam(model_g2.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    X_t = torch.tensor(X_all, device=device)
    y_t = torch.tensor(y_all, device=device)

    losses = []
    n_epochs = 200
    batch_size = 512
    N = len(X_t)

    for epoch in range(n_epochs):
        perm = torch.randperm(N, device=device)
        epoch_loss = 0.0
        for start in range(0, N, batch_size):
            idx = perm[start: start + batch_size]
            pred = model_g2(X_t[idx])
            loss = loss_fn(pred, y_t[idx])
            opt.zero_grad()
            loss.backward()
            opt.step()
            epoch_loss += loss.item() * len(idx)
        losses.append(epoch_loss / N)
        if (epoch + 1) % 50 == 0:
            print(f'Epoch {epoch+1:3d}/{n_epochs}  loss={losses[-1]:.6f}')

    plt.plot(losses)
    plt.xlabel('Epoch')
    plt.ylabel('MSE loss')
    plt.title('G₂: ρ(r) → c⁽¹⁾(r)  training')
    plt.yscale('log')
    plt.tight_layout()
    plt.show()

    # Quick sanity: predict c1 on a held-out sample
    s_test = load_sample(sample_files[-1])
    X_test, y_test = make_windows(s_test['rho'], s_test['c1'])
    with torch.no_grad():
        pred_c1 = model_g2(torch.tensor(X_test, device=device)).cpu().numpy()

    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot(s_test['z'], y_test[:len(s_test['z'])], label='GCMC c⁽¹⁾', lw=2)
    ax.plot(s_test['z'], pred_c1[:len(s_test['z'])], '--', label='G₂ prediction')
    ax.set_xlabel('z [Å]')
    ax.set_ylabel('c⁽¹⁾(z)')
    ax.legend()
    plt.tight_layout()
    plt.show()

## CLI usage

```bash
# Run a full campaign (all parameters in YAML)
mlip_mc --mode dft-data \
        --campaign-config configs/campaign_sammuller.yaml \
        --model hf://your-org/your-mace-model \
        --output-dir ./cdft_data

# Override μ and number of runs from CLI
mlip_mc --mode dft-data \
        --campaign-config configs/campaign_sammuller.yaml \
        --model path/to/model.pt \
        --mu-values "-0.5,-0.3,-0.1,0.1" \
        --n-runs 100 \
        --output-dir ./cdft_data
```